## Setup

In [1]:
import os, sys, json, time, copy, shutil, zipfile, warnings
import numpy as np, pandas as pd, cv2
import tensorflow as tf
try: tf.config.set_visible_devices([], 'GPU')          # Force CPU so the timings match the deployment
except Exception as e: print('GPU hiding skipped:', e)
from tensorflow import keras
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')
print('TensorFlow', tf.__version__, '| visible GPUs:', tf.config.list_physical_devices('GPU'), '| CPU cores:', os.cpu_count())

from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/SignSense_data'; WORK, PROC, OUT, MODELS = '/content/work', '/content/processed', '/content/results', '/content/results/models'
for d in (WORK, OUT, MODELS): os.makedirs(d, exist_ok=True)
for f in ['processed_metadata.csv', 'label_map.json', 'results/static_test_predictions.npz', 'results/pipeline_config.json']: shutil.copy(f'{DRIVE_DIR}/{f}', WORK)

meta = pd.read_csv(f'{WORK}/processed_metadata.csv'); imgs = meta[meta['media_type'] == 'image'].reset_index(drop=True)
te_df = imgs[imgs['split'] == 'test'].reset_index(drop=True)
SP = np.load(f'{WORK}/static_test_predictions.npz', allow_pickle=True); assert (SP['y'] == te_df['class_id'].values).all()
pcfg = json.load(open(f'{WORK}/pipeline_config.json')); print('Pipeline config loaded:', pcfg['typer'])

TensorFlow 2.21.0 | visible GPUs: [] | CPU cores: 2
Mounted at /content/drive
Pipeline config loaded: {'letter_conf': 0.5, 'stable_frames': 3, 'dynamic_conf': 0.6, 'control_conf': 0.83, 'control_frames': 8, 'neutral_frames': 3, 'require_neutral': 'same', 'correct_words': True, 'max_dist': 1}


In [2]:
# 300 test images for the parity check and the timing
idx = np.random.default_rng(0).choice(len(te_df), 300, replace=False)
need_files = te_df.loc[idx, 'processed_path'].tolist()
if not all(os.path.exists(os.path.join(PROC, p)) for p in need_files):
    zp = f'{DRIVE_DIR}/processed_images.zip'; shutil.copy(zp, '/content/processed_images.zip')
    with zipfile.ZipFile('/content/processed_images.zip') as z:
        for p in need_files: z.extract(p, PROC)
X = np.stack([cv2.cvtColor(cv2.imread(os.path.join(PROC, p)), cv2.COLOR_BGR2RGB) for p in need_files]).astype('float32'); print(X.shape)

(300, 224, 224, 3)


## 1. Convert to float32 and check the predictions are unchanged

In [3]:
def to_fp32_config(obj):
    # Recursively replace every mixed_float16 dtype policy in a Keras config by float32
    if isinstance(obj, dict):
        if obj.get('class_name') == 'DTypePolicy' and obj.get('config', {}).get('name') == 'mixed_float16':
            return {**obj, 'config': {**obj['config'], 'name': 'float32'}}
        return {k: to_fp32_config(v) for k, v in obj.items()}
    if isinstance(obj, list): return [to_fp32_config(v) for v in obj]
    return obj

FILES = {'Baseline CNN': 'baseline_cnn', 'MobileNetV2': 'mobilenetv2', 'EfficientNet-B0': 'efficientnetb0'}
y_true = te_df.loc[idx, 'class_id'].values
FP32, REJECTED, PARITY = {}, {}, []
for name, f in FILES.items():
    src = keras.models.load_model(f'{DRIVE_DIR}/models/{f}.keras', compile=False)
    m32 = keras.Model.from_config(to_fp32_config(src.get_config())); m32.set_weights(src.get_weights())
    p32 = m32.predict(X, batch_size=32, verbose=0); pref = SP[name.replace(' ', '_')][idx].astype('float32')   # pref = probabilities saved at evaluation time
    p_src = src.predict(X, batch_size=32, verbose=0)                                                          # the original model run again on this machine
    a32, aref, asrc = (p32.argmax(1) == y_true).mean(), (pref.argmax(1) == y_true).mean(), (p_src.argmax(1) == y_true).mean()
    dis = np.where(p32.argmax(1) != pref.argmax(1))[0]
    ok = bool(a32 >= aref - 0.02)       # Rule: the float32 copy may not be more than 2 points less accurate than the original
    PARITY.append({'model': name, 'acc_original_saved': round(float(aref), 4), 'acc_original_rerun': round(float(asrc), 4), 'acc_float32': round(float(a32), 4),
                   'argmax_agreement': round(float((p32.argmax(1) == pref.argmax(1)).mean()), 4), 'max_abs_prob_diff': round(float(np.abs(p32 - pref).max()), 5),
                   'disagreements': int(len(dis)), 'float32_right_on_them': int((p32.argmax(1)[dis] == y_true[dis]).sum()), 'original_right_on_them': int((pref.argmax(1)[dis] == y_true[dis]).sum()), 'accepted': ok})
    if ok: m32.save(f'{MODELS}/{f}_fp32.keras'); FP32[name] = m32
    else: REJECTED[name] = m32
    del src
par = pd.DataFrame(PARITY); display(par)
print('Accepted float32 models:', list(FP32), '| rejected:', list(REJECTED))
assert 'MobileNetV2' in FP32, 'MobileNetV2 float32 conversion failed the accuracy check'

,model,acc_original_saved,acc_original_rerun,acc_float32,argmax_agreement,max_abs_prob_diff,disagreements,float32_right_on_them,original_right_on_them,accepted
0,Baseline CNN,0.7767,0.7767,0.7767,1.0000,0.00604,0,0,0,True
1,MobileNetV2,0.8567,0.8567,0.8567,1.0000,0.00024,0,0,0,True
2,EfficientNet-B0,0.8200,0.8200,0.8267,0.9333,0.82773,20,8,6,True


Accepted float32 models: ['Baseline CNN', 'MobileNetV2', 'EfficientNet-B0'] | rejected: []


**What it tells us:** a good conversion gives the same decisions as the original (agreement near 100%). If a model disagrees, the table shows who is right on the images where they differ: the float32 copy or the original float16 model. **Acceptance rule (fixed in advance):** a float32 model is accepted if its accuracy on these 300 images is at most 2 points below the original; a model that fails is left out of the app. EfficientNet is known to be sensitive to float16, so a disagreement can come from the original and not from the conversion.

## 2. CPU inference time per frame

In [4]:
def cpu_ms(model, n=60):
    x1 = tf.constant(X[:1]); fn = tf.function(lambda a: model(a, training=False))
    for _ in range(8): fn(x1).numpy()
    t = time.time()
    for _ in range(n): fn(x1).numpy()
    return (time.time() - t) / n * 1000
LAT = {n: cpu_ms(m) for n, m in FP32.items()}
rows = [{'option': n, 'cpu_ms_per_frame': round(v, 1), 'params_M': round(FP32[n].count_params() / 1e6, 2), 'file_MB': round(os.path.getsize(f'{MODELS}/{FILES[n]}_fp32.keras') / 1e6, 1)} for n, v in LAT.items()]
if len(FP32) == 3: rows.append({'option': 'Ensemble (3 models)', 'cpu_ms_per_frame': round(sum(LAT.values()), 1), 'params_M': round(sum(m.count_params() for m in FP32.values()) / 1e6, 2), 'file_MB': round(sum(r['file_MB'] for r in rows), 1)})
lat = pd.DataFrame(rows); lat['frames_per_second'] = (1000 / lat['cpu_ms_per_frame']).round(1); display(lat)

,option,cpu_ms_per_frame,params_M,file_MB,frames_per_second
0,Baseline CNN,99.0,1.03,4.3,10.1
1,MobileNetV2,55.3,2.30,9.8,18.1
2,EfficientNet-B0,67.6,4.10,17.2,14.8
3,Ensemble (3 models),222.0,7.43,31.3,4.5


**What it tells us:** these are the real per-frame costs on a CPU like the deployment machine. The ensemble runs its three models one after the other, so its time is their sum.

## 3. Choose the app configuration (rule fixed above)

In [5]:
BUDGET_MS = 250
mob = float(lat.loc[lat.option == 'MobileNetV2', 'cpu_ms_per_frame'].iloc[0]); ens = float(lat.loc[lat.option == 'Ensemble (3 models)', 'cpu_ms_per_frame'].iloc[0]) if (lat.option == 'Ensemble (3 models)').any() else float('inf')
if ens <= BUDGET_MS: STATIC_MODE, STATIC_FILES = 'ensemble', [f'{v}_fp32.keras' for v in FILES.values()]
else: STATIC_MODE, STATIC_FILES = 'mobilenetv2', ['mobilenetv2_fp32.keras']
print(f'Ensemble {ens:.0f} ms (inf = a member was rejected) vs budget {BUDGET_MS} ms -> chosen static setup: {STATIC_MODE}  (MobileNetV2 alone: {mob:.0f} ms)')

typer = dict(pcfg['typer']); typer['stable_frames'] = 5
app_cfg = {'static': {'mode': STATIC_MODE, 'files': STATIC_FILES, 'input_size': 224, 'pad_fill_rgb': [175, 159, 153], 'cpu_ms_per_frame': round(ens if STATIC_MODE == 'ensemble' else mob, 1), 'rejected_float32_models': list(REJECTED)},
           'dynamic': {'mode': 'push_to_record', 'file': 'dynamic_rf.joblib', 'frames': 88, 'max_record_seconds': 4},
           'control': {'file': 'control_gesture_clf.joblib', 'threshold': pcfg['ctrl_threshold'], 'classes': ['delete', 'other', 'space']},
           'typer': typer, 'lexicon_file': 'urdu_lexicon.txt', 'engine_file': 'typing_engine.py',
           'landmarker_model_url': 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task',
           'notes': 'Automatic motion routing was evaluated and rejected (13/59 dynamic test videos found); dynamic letters use push-to-record.'}
json.dump(app_cfg, open(f'{OUT}/app_config.json', 'w'), indent=1); lat.to_csv(f'{OUT}/static_cpu_latency.csv', index=False); par.to_csv(f'{OUT}/float32_parity.csv', index=False)
os.makedirs(f'{DRIVE_DIR}/models', exist_ok=True)
for f in os.listdir(MODELS): shutil.copy(f'{MODELS}/{f}', f'{DRIVE_DIR}/models/{f}')
for f in os.listdir(OUT):
    if os.path.isfile(f'{OUT}/{f}'): shutil.copy(f'{OUT}/{f}', f'{DRIVE_DIR}/results/{f}')
print(json.dumps(app_cfg['typer'], indent=1)); print(sorted(os.listdir(OUT)), sorted(os.listdir(MODELS)))

Ensemble 222 ms (inf = a member was rejected) vs budget 250 ms -> chosen static setup: ensemble  (MobileNetV2 alone: 55 ms)
{
 "letter_conf": 0.5,
 "stable_frames": 5,
 "dynamic_conf": 0.6,
 "control_conf": 0.83,
 "control_frames": 8,
 "neutral_frames": 3,
 "require_neutral": "same",
 "correct_words": true,
 "max_dist": 1
}
['app_config.json', 'float32_parity.csv', 'models', 'static_cpu_latency.csv'] ['baseline_cnn_fp32.keras', 'efficientnetb0_fp32.keras', 'mobilenetv2_fp32.keras']


## 4. Summary

In [6]:
def hdr(t): print('\n' + '=' * 70 + f'\n{t}\n' + '=' * 70)
hdr('Float32 parity'); print(par.to_string(index=False)); print('Rejected:', list(REJECTED))
hdr('CPU latency'); print(lat.to_string(index=False)); print('CPU cores:', os.cpu_count())
hdr('Chosen static setup'); print(STATIC_MODE, STATIC_FILES)
hdr('Files'); print(sorted(os.listdir(OUT)), sorted(os.listdir(MODELS)))


Float32 parity
          model  acc_original_saved  acc_original_rerun  acc_float32  argmax_agreement  max_abs_prob_diff  disagreements  float32_right_on_them  original_right_on_them  accepted
   Baseline CNN              0.7767              0.7767       0.7767            1.0000            0.00604              0                      0                       0      True
    MobileNetV2              0.8567              0.8567       0.8567            1.0000            0.00024              0                      0                       0      True
EfficientNet-B0              0.8200              0.8200       0.8267            0.9333            0.82773             20                      8                       6      True
Rejected: []

CPU latency
             option  cpu_ms_per_frame  params_M  file_MB  frames_per_second
       Baseline CNN              99.0      1.03      4.3               10.1
        MobileNetV2              55.3      2.30      9.8               18.1
    EfficientNet-B